In [ ]:
# install required packages
%pip install numpy mne osl-dynamics tensorflow ipyevents ipywidgets pyvista pyvistaqt pandas seaborn nibabel nilearn scikit-image numba pyyaml scikit-learn
%pip install mne pymatreader

In [ ]:
# import packages
import sys
!conda run -n hmmnet pip install h5io

import mne
import numpy as np
import scipy.io
import h5py
from pymatreader import read_mat

from osl_dynamics import analysis, inference, files
from osl_dynamics.meeg import preproc, rhino, source_recon, parcellation
from osl_dynamics.data import Data
from osl_dynamics.utils import plotting
from osl_dynamics.utils.filenames import OSLFilenames

import copy
import argparse
import csv
import os
import re
import sys
from collections import Counter
from scipy import signal
from scipy.spatial import cKDTree


In [ ]:
# load Epoched EMG/MEG data from .ds file
subject_folder = '//rdp.arc.ucl.ac.uk/ritd-ag-project-rd01pz-dbush99/Katja Button Press Data/s05/'
subject_filename = 'Subject5_spm_mne'

filepath = subject_folder + subject_filename + '.mat'

In [ ]:
# load the .mat file using h5py
with h5py.File(filepath, 'r') as f:
    print(list(f.keys()))
    data       = np.array(f['data']).T      

    # change to MNE's expected format: (epochs, channels, time points)
    epochs_data = np.asarray(data).transpose(2, 0, 1)
    
    sfreq      = float(np.array(f['fs']).flatten()[0])
    chan_names = []

    for ref in f['chan_names'][:, 0]:
        name = ''.join(chr(c) for c in f[ref][:].flatten())
        chan_names.append(name)


In [ ]:
# check data and channel info
print(f"Data shape: {data.shape}")
print(f"N channels: {len(chan_names)}")
print(f"Sampling rate: {sfreq} Hz")
print(f"Channel names: {chan_names}")

In [ ]:
# assign information about the channels and sampling frequency
# EEG-prefixed labels contain EMG data; CTF Omega 275 M channels are gradiometers.
channel_types = [
    "emg" if name.upper().startswith("EEG") else
    "grad" if name.upper().startswith("M") else
    "misc"
    for name in chan_names
]
info = mne.create_info(
    ch_names=chan_names,
    sfreq=sfreq,
    ch_types=channel_types,
)


In [ ]:
# Read sensor metadata while the HDF5 file is open. MNE can store channel
# positions and head fiducials in a DigMontage. skip if channel positions are not needed

def _read_h5_value(obj):
    if isinstance(obj, h5py.Group):
        return {key: _read_h5_value(value) for key, value in obj.items()}

    value = obj[()]
    if isinstance(value, np.ndarray) and value.dtype == h5py.ref_dtype:
        return [_read_h5_value(obj.file[ref]) for ref in value.ravel() if ref]

    value = np.asarray(value)
    if value.dtype.kind in "ui" and value.ndim <= 2:
        # MATLAB character arrays are commonly stored as integer character codes.
        try:
            text = "".join(chr(int(c)) for c in value.ravel() if c)
            if text and all(c.isprintable() for c in text):
                return text
        except (ValueError, TypeError):
            pass
    return value



# adds CTF channel metadata to the MNE Info object when a matching CTF dataset is found in the same folder as the .mat file.
def _find_field(obj, names):
    if isinstance(obj, dict):
        for key, value in obj.items():
            if key.lower() in names:
                ctf_path = os.path.join(subject_folder, "mg4670_katja2_20160623_01.ds")
                if os.path.isdir(ctf_path):
                    ctf_raw = mne.io.read_raw_ctf(ctf_path, preload=False, verbose="ERROR")
                    try:
                        ctf_info = ctf_raw.info
                        target_names = info["ch_names"]
                        target_by_name = {name.casefold(): i for i, name in enumerate(target_names)}

                        # Copy channel types from CTF for every matching channel.
                        channel_types = {}
                        channel_pairs = []
                        for ctf_idx, name in enumerate(ctf_info["ch_names"]):
                            target_idx = target_by_name.get(name.casefold())
                            if target_idx is None:
                                continue

                            target_name = target_names[target_idx]
                            ctf_type = mne.channel_type(ctf_info, ctf_idx)
                            channel_types[target_name] = "emg" if target_name.upper().startswith("EEG") else ctf_type
                            channel_pairs.append((ctf_idx, target_idx))

                        if channel_types:
                            info.set_channel_types(channel_types, on_unit_change="ignore")

                            # Info channel dictionaries are locked by MNE; unlock to copy
                            # geometry and coil/unit metadata from the CTF dataset.
                            with info._unlock():
                                for ctf_idx, target_idx in channel_pairs:
                                    if mne.channel_type(ctf_info, ctf_idx) in {"mag", "grad", "ref_meg"}:
                                        for key in ("loc", "coil_type", "coord_frame", "unit", "unit_mul"):
                                            info["chs"][target_idx][key] = copy.deepcopy(
                                                ctf_info["chs"][ctf_idx][key]
                                            )

                                # Compensation matrices are only safe to copy when channel
                                # order is identical in both datasets.
                                if target_names == ctf_info["ch_names"]:
                                    info["comps"] = copy.deepcopy(ctf_info["comps"])
                                    info["dev_head_t"] = copy.deepcopy(ctf_info["dev_head_t"])

                            print(f"Added CTF metadata for {len(channel_pairs)} matching channels.")
                        else:
                            print("No matching channels found in the CTF dataset.")
                    finally:
                        ctf_raw.close()
                else:
                    print(f"CTF dataset not found: {ctf_path}")
        for value in obj.values():
            found = _find_field(value, names)
            if found is not None:
                return found
    return None


# load sens data from the .mat file to extract channel positions and fiducials
with h5py.File(filepath, "r") as mat:
    sens = _read_h5_value(mat["sens"])

# Look for common sensor-structure field names.
pos = _find_field(sens, {"chanpos", "elecpos", "positions", "position", "pos"})
labels = _find_field(sens, {"label", "labels", "chan_names", "ch_names"})
nasion = _find_field(sens, {"nasion", "nas"})
lpa = _find_field(sens, {"lpa", "left_preauricular"})
rpa = _find_field(sens, {"rpa", "right_preauricular"})
coord_frame = _find_field(sens, {"coord_frame", "coordinate_frame", "frame"})

# Convert various data types to a list of strings for channel labels.
def _as_text_list(value):
    if isinstance(value, str):
        return [value]
    if isinstance(value, list):
        return [str(x) for x in value]
    if isinstance(value, np.ndarray):
        return [str(x) for x in value.ravel()]
    return None

# Convert various data types to a 2D array of shape (n_channels, 3) for channel positions.
def _xyz(value):
    if value is None:
        return None
    arr = np.asarray(value, dtype=float).squeeze()
    if arr.ndim == 1 and arr.size == 3:
        arr = arr.reshape(1, 3)
    elif arr.ndim == 2 and arr.shape[0] == 3 and arr.shape[1] != 3:
        arr = arr.T
    if arr.ndim != 2 or arr.shape[1] != 3:
        return None
    # Convert coordinates that appear to be in millimetres to metres.
    if np.nanmax(np.abs(arr)) > 2:
        arr = arr / 1000.0
    return arr

positions = _xyz(pos)
sensor_labels = _as_text_list(labels)

# Only attach coordinates when their correspondence to actual EEG channels is clear.
if positions is not None:
    if sensor_labels and len(sensor_labels) == len(positions):
        position_by_name = dict(zip(sensor_labels, positions))
    elif len(positions) == len(chan_names):
        position_by_name = dict(zip(chan_names, positions))
    else:
        position_by_name = {}

    eeg_positions = {
        name: position_by_name[name]
        for idx, name in enumerate(chan_names)
        if mne.channel_type(info, idx) == "eeg" and name in position_by_name
    }

    if eeg_positions:
        frame_text = str(coord_frame).lower() if coord_frame is not None else ""
        if "head" in frame_text:
            montage = mne.channels.make_dig_montage(
                ch_pos=eeg_positions,
                nasion=_xyz(nasion)[0] if _xyz(nasion) is not None else None,
                lpa=_xyz(lpa)[0] if _xyz(lpa) is not None else None,
                rpa=_xyz(rpa)[0] if _xyz(rpa) is not None else None,
                coord_frame="head",
            )
            info.set_montage(montage, on_missing="ignore")
            print(f"Attached positions for {len(eeg_positions)} EEG channels and available fiducials.")
        else:
            print("Sensor positions found, but their coordinate frame is not identified as head; not attached.")
    else:
        print("No matching EEG channel positions found in sens.")
else:
    print("No recognized channel-position field found in sens.")

# Keep an explicit name/type record alongside the channel names.
channel_metadata = [
    {"name": name, "type": mne.channel_type(info, idx)}
    for idx, name in enumerate(info["ch_names"])
]


In [ ]:
# make MNE EpochsArray for data (with shape (n_epochs, n_channels, n_times))
epochs = mne.EpochsArray(epochs_data, info)

print(epochs)
print("Epoch data shape:", epochs.get_data().shape)

In [ ]:
# save the MNE Epochs object to a .fif file on the subject folder
os.chdir(subject_folder)        
outfile = os.path.join(subject_folder, f"{subject_filename}-epo.fif")
epochs.save(outfile, overwrite=True, verbose=False)
print(f"Saved to: {outfile}")


In [ ]:
# Save to a writable local folder
out_dir = os.path.join(os.path.expanduser("~"), "mne_exports")
os.makedirs(out_dir, exist_ok=True)
outfile = os.path.join(out_dir, f"{subject_filename}_EEG.fif")
epochs.save(outfile, overwrite=True, verbose=False)
print(f"Saved to: {outfile}")